In [3]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


# AQI Historical Data Collection

This notebook collects historical air pollution observations for Nagarparkar using the OpenWeather Air Pollution API.

The previous data collection process returned only one observation, which was insufficient for training and evaluating a machine learning forecasting model.

Therefore, this pipeline collects multiple historical observations containing AQI and major pollutant measurements. The collected dataset will be used in the subsequent preprocessing, feature engineering, exploratory analysis, and model training stages.

In [4]:
import os
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv

## 1. Project Configuration

The geographic coordinates of Nagarparkar are used to request location-specific air pollution observations.

The API key is loaded from the environment instead of being written directly in the notebook.

In [5]:
load_dotenv()

API_KEY = os.getenv("OPENWEATHER_API_KEY")

CITY = "Nagarparkar"
LATITUDE = 24.357
LONGITUDE = 70.755

if not API_KEY:
    raise ValueError("OPENWEATHER_API_KEY was not found.")

print("City:", CITY)
print("Latitude:", LATITUDE)
print("Longitude:", LONGITUDE)

City: Nagarparkar
Latitude: 24.357
Longitude: 70.755


## 2. Define Historical Collection Period

A forecasting model requires multiple observations over time.

A historical time window is defined to collect air pollution measurements from previous days. This provides a chronological dataset instead of a single real-time observation.

In [7]:
DAYS_BACK = 5

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=DAYS_BACK)

start_timestamp = int(start_time.timestamp())
end_timestamp = int(end_time.timestamp())

print("Historical data period:")
print("Start:", start_time)
print("End:", end_time)
print("Days requested:", DAYS_BACK)

Historical data period:
Start: 2026-08-30 02:56:45.383288+00:00
End: 2026-09-04 02:56:45.383288+00:00
Days requested: 5


## 3. Request Historical Air Pollution Data

The historical air pollution endpoint is queried using the selected geographic coordinates and time range.

The API response contains observations with an AQI category and pollutant concentrations.

In [8]:
url = "https://api.openweathermap.org/data/2.5/air_pollution/history"

params = {
    "lat": LATITUDE,
    "lon": LONGITUDE,
    "start": start_timestamp,
    "end": end_timestamp,
    "appid": API_KEY
}

response = requests.get(url, params=params, timeout=30)

print("API Status Code:", response.status_code)

if response.status_code != 200:
    print(response.text)
    response.raise_for_status()

data = response.json()

print("Response received successfully.")

API Status Code: 200
Response received successfully.


## 4. Convert API Response into Tabular Data

The JSON response is transformed into a structured pandas DataFrame.

Each observation contains the timestamp, AQI value, and concentrations of the major air pollutants required for the machine learning pipeline.

In [9]:
records = []

for item in data.get("list", []):

    components = item.get("components", {})
    main = item.get("main", {})

    record = {
        "datetime": pd.to_datetime(item["dt"], unit="s"),
        "AQI": main.get("aqi"),
        "CO": components.get("co"),
        "NO": components.get("no"),
        "NO2": components.get("no2"),
        "O3": components.get("o3"),
        "SO2": components.get("so2"),
        "PM2_5": components.get("pm2_5"),
        "PM10": components.get("pm10"),
        "NH3": components.get("nh3")
    }

    records.append(record)

df = pd.DataFrame(records)

if df.empty:
    raise ValueError("No historical observations were returned by the API.")

print("Observations collected:", len(df))
print("Dataset shape:", df.shape)

display(df.head())

Observations collected: 120
Dataset shape: (120, 10)


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
0,2026-08-30 03:00:00,2,70.64,0.05,1.18,35.02,0.90,6.99,20.53,0.00
1,2026-08-30 04:00:00,1,71.37,0.13,1.16,36.15,0.96,6.50,18.09,0.00
2,2026-08-30 05:00:00,1,71.14,0.17,0.98,38.45,0.94,6.11,15.69,0.00
3,2026-08-30 06:00:00,1,70.22,0.16,0.77,41.43,0.91,6.07,14.75,0.01
4,2026-08-30 07:00:00,1,70.41,0.11,0.71,43.64,0.87,6.01,15.01,0.00


## 5. Validate and Prepare the Raw Dataset

Before saving the dataset, duplicate timestamps are removed and observations are ordered chronologically.

Basic validation is performed to inspect missing values, duplicates, and the temporal range of the collected data.

In [11]:
df["datetime"] = pd.to_datetime(df["datetime"])

df = (
    df
    .drop_duplicates(subset=["datetime"])
    .sort_values("datetime")
    .reset_index(drop=True)
)

numeric_columns = [
    "AQI",
    "CO",
    "NO",
    "NO2",
    "O3",
    "SO2",
    "PM2_5",
    "PM10",
    "NH3"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Final dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate timestamps:")
print(df["datetime"].duplicated().sum())

print("\nDate range:")
print("Start:", df["datetime"].min())
print("End:", df["datetime"].max())

display(df.head())

Final dataset shape: (120, 10)

Missing values:
datetime    0
AQI         0
CO          0
NO          0
NO2         0
O3          0
SO2         0
PM2_5       0
PM10        0
NH3         0
dtype: int64

Duplicate timestamps:
0

Date range:
Start: 2026-08-30 03:00:00
End: 2026-09-04 02:00:00


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
0,2026-08-30 03:00:00,2,70.64,0.05,1.18,35.02,0.90,6.99,20.53,0.00
1,2026-08-30 04:00:00,1,71.37,0.13,1.16,36.15,0.96,6.50,18.09,0.00
2,2026-08-30 05:00:00,1,71.14,0.17,0.98,38.45,0.94,6.11,15.69,0.00
3,2026-08-30 06:00:00,1,70.22,0.16,0.77,41.43,0.91,6.07,14.75,0.01
4,2026-08-30 07:00:00,1,70.41,0.11,0.71,43.64,0.87,6.01,15.01,0.00


## 6. Save Raw Historical Dataset

The validated historical dataset is saved in the raw data directory.

This dataset becomes the input for the preprocessing stage of the AQI forecasting pipeline.

In [12]:
output_path = "../data/raw/aqi_raw_data.csv"

os.makedirs("../data/raw", exist_ok=True)

df.to_csv(output_path, index=False)

print("Raw historical dataset saved successfully.")
print("Path:", output_path)
print("Total observations:", len(df))

Raw historical dataset saved successfully.
Path: ../data/raw/aqi_raw_data.csv
Total observations: 120


## 7. Final Dataset Verification

The saved dataset is loaded again to confirm that the collection pipeline completed successfully and that the data is available for the next preprocessing stage.

In [13]:
saved_df = pd.read_csv("../data/raw/aqi_raw_data.csv")

print("Saved dataset shape:", saved_df.shape)

print("\nColumns:")
print(saved_df.columns.tolist())

print("\nFirst 5 observations:")
display(saved_df.head())

print("\nLast 5 observations:")
display(saved_df.tail())

Saved dataset shape: (120, 10)

Columns:
['datetime', 'AQI', 'CO', 'NO', 'NO2', 'O3', 'SO2', 'PM2_5', 'PM10', 'NH3']

First 5 observations:


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
0,2026-08-30 03:00:00,2,70.64,0.05,1.18,35.02,0.90,6.99,20.53,0.00
1,2026-08-30 04:00:00,1,71.37,0.13,1.16,36.15,0.96,6.50,18.09,0.00
2,2026-08-30 05:00:00,1,71.14,0.17,0.98,38.45,0.94,6.11,15.69,0.00
3,2026-08-30 06:00:00,1,70.22,0.16,0.77,41.43,0.91,6.07,14.75,0.01
4,2026-08-30 07:00:00,1,70.41,0.11,0.71,43.64,0.87,6.01,15.01,0.00



Last 5 observations:


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
115,2026-09-03 22:00:00,1,69.87,0.00,0.96,34.62,0.78,5.28,13.57,0.0
116,2026-09-03 23:00:00,1,70.22,0.00,0.89,35.06,0.74,5.23,13.99,0.0
117,2026-09-04 00:00:00,1,70.86,0.00,0.89,35.42,0.74,5.20,14.37,0.0
118,2026-09-04 01:00:00,1,71.96,0.00,1.02,35.57,0.78,5.18,14.70,0.0
119,2026-09-04 02:00:00,1,74.20,0.01,1.39,35.42,0.91,5.16,14.89,0.0
